# Ensemble i Global LightGBM

En l'avaluació, cap model individual no supera MASE=1 de forma consistent sobre totes les tecnologies i tots els folds. Dues estratègies complementàries poden millorar el rendiment sense incorporar dades externes:

**1. Ensemble (ETS Additiu + AutoSARIMA + TBATS).** La combinació simple (mitjana aritmètica) de les prediccions dels tres millors models estadístics del benchmark esmorteeix els errors individuals. La intuïció és directa: cada model comet errors en diferent direcció. L'ETS Additiu és estable però subestima els pics en sèries amb variança creixent. AutoSARIMA captura bé l'autocorrelació però pot ser rígid davant canvis de nivell. TBATS compensa via la transformació Box-Cox però és menys precís en sèries amb poca variança. La seva combinació, documentada com a millor pràctica a les competicions M (Makridakis et al., 2019), redueix l'error esperat sense requerir cap dada addicional ni cap reentrenament.

**2. Global LightGBM (cross-series).** El problema del LightGBM individual era l'escassetat de dades: N≈84 observacions per sèrie no és suficient per aprendre l'estacionalitat de forma implícita via gradient boosting. Si s'entrena un sol model sobre les 7 sèries simultàniament s'obtenen 7×84≈588 observacions d'entrenament — set vegades més. El model pot aprendre patrons compartits d'estacionalitat i tendència (tots els vehicles matriculats a Catalunya segueixen un cicle anual similar) i al mateix temps distingir les tecnologies via una variable `tech_id` categòrica (valors 0–6). Mantenem l'estratègia *direct multi-step*: un model independent per a cada horitzó h=1..12.

> **Prerequisit:** executar tots els notebooks de models anteriors (baseline.ipynb, sarima.ipynb, tbats.ipynb, lgbm.ipynb), ja que l'Ensemble carrega els seus CSV de resultats.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from matplotlib.lines import Line2D

from features import TECNOLOGIES_MODELITZABLES, build_ml_features, get_feature_columns
from evaluation import compute_metrics, walk_forward_cv_full
from plotting import format_ts_ax, plot_backtest_per_tech, plot_wfv_per_tech

DATA_DIR    = Path('../data/processed')
RESULTS_DIR = DATA_DIR / 'results'
FIGURES_DIR = Path('../figures')

plt.rcParams.update({'figure.dpi': 120})
TECNOLOGIES    = TECNOLOGIES_MODELITZABLES + ['Total']
H              = 12
FEATURE_COLS   = get_feature_columns()
FEATURE_COLS_G = FEATURE_COLS + ['tech_id']   # global model: afegeix identificador de sèrie

LGB_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    verbose=-1,
)

print(f'Features per sèrie: {len(FEATURE_COLS)} | Features model global: {len(FEATURE_COLS_G)}')
print(f'Tecnologies: {TECNOLOGIES}')

In [ ]:
pivot_train = pd.read_csv(DATA_DIR / 'pivot_train.csv', index_col='date', parse_dates=True)
pivot_test  = pd.read_csv(DATA_DIR / 'pivot_test.csv',  index_col='date', parse_dates=True)
pivot_full  = pd.concat([pivot_train, pivot_test]).sort_index()
train_set   = pd.read_csv(DATA_DIR / 'train_set.csv',   parse_dates=['date'])

print(f'Train: {pivot_train.index[0].date()} → {pivot_train.index[-1].date()} ({len(pivot_train)} mesos)')
print(f'Test:  {pivot_test.index[0].date()}  → {pivot_test.index[-1].date()}  ({len(pivot_test)} mesos)')
print(f'Sèrie completa: {len(pivot_full)} mesos')

---
## Secció 1: Ensemble (ETS Additiu + AutoSARIMA + TBATS)

### 1.1 Metodologia

L'ensemble que proposem és la **mitjana aritmètica simple** (equal-weight average) de les prediccions dels tres millors models del benchmark, ordenats per MASE WFV:

| Component | WFV MASE (benchmark) | Rol en l'ensemble |
|---|---|---|
| ETS Additiu | 1.01 | Estabilitat global, bon nivell base |
| TBATS | 1.11 | Box-Cox estabilitza variança, bo al Total |
| AutoSARIMA | 1.14 | Captura autocorrelació, millor a Gasolina i Diesel |

No s'usen pesos optimitzats (stacking) per evitar l'overfitting: amb 13 folds i 7 sèries, qualsevol optimització dels pesos ajustaria el soroll específic dels folds en lloc d'aprendre la millor combinació real. La literatura de les M-competitions confirma que equal-weight supera pesos optimitzats en la majoria de casos quan N és petit.

Les prediccions de cada component ja estan calculades i guardades en CSV. L'ensemble no requereix cap nou entrenament: només combinar els resultats existents.

In [ ]:
# Carregar forecasts dels 3 components
fc_baseline = pd.read_csv(RESULTS_DIR / 'baseline_forecasts.csv', parse_dates=['date'])
fc_sarima   = pd.read_csv(RESULTS_DIR / 'sarima_forecasts.csv',   parse_dates=['date'])
fc_tbats    = pd.read_csv(RESULTS_DIR / 'tbats_forecasts.csv',    parse_dates=['date'])

fc_ets = fc_baseline[fc_baseline['Model'] == 'ETS_Additiu'].copy()
fc_sar = fc_sarima[fc_sarima['Model'] == 'AutoSARIMA'].copy()
fc_tbt = fc_tbats[fc_tbats['Model'] == 'TBATS'].copy()

# Unir per (Tecnologia, date) i fer la mitjana
key = ['Tecnologia', 'date']
ens = (fc_ets[key + ['predicted', 'actual']]
       .rename(columns={'predicted': 'pred_ets'})
       .merge(fc_sar[key + ['predicted']].rename(columns={'predicted': 'pred_sar'}), on=key)
       .merge(fc_tbt[key + ['predicted']].rename(columns={'predicted': 'pred_tbt'}), on=key))
ens['predicted'] = (ens['pred_ets'] + ens['pred_sar'] + ens['pred_tbt']) / 3
ens['Model']     = 'Ensemble'
df_forecasts_ens = ens[key + ['Model', 'predicted', 'actual']].copy()

print(f'Forecasts ensemble: {len(df_forecasts_ens)} registres ({len(df_forecasts_ens)//len(TECNOLOGIES)} mesos × {len(TECNOLOGIES)} tecnologies)')
print(f'Exemple (Total, primers 3 mesos):')
display(df_forecasts_ens[df_forecasts_ens['Tecnologia'] == 'Total'].head(3)[['date','predicted','actual']])

### 1.2 Mètriques sobre el test (backtesting 2025)

In [ ]:
metrics_rows = []
for tech in TECNOLOGIES:
    sub = df_forecasts_ens[df_forecasts_ens['Tecnologia'] == tech].sort_values('date')
    y_true  = sub['actual'].values
    y_pred  = sub['predicted'].values
    y_train = pivot_train[tech].dropna().values
    m = compute_metrics(y_true, y_pred, y_train)
    m.update({'Tecnologia': tech, 'Model': 'Ensemble'})
    metrics_rows.append(m)
    print(f'  {tech:35s}: RMSE={m["RMSE"]:.0f}, MAPE={m["MAPE"]:.1f}%, MASE={m["MASE"]:.2f}')

df_metrics_ens = pd.DataFrame(metrics_rows)

sub_view = df_metrics_ens[df_metrics_ens['Tecnologia'] != 'Total'].set_index('Tecnologia')
print('\nMètriques Ensemble per sub-tecnologia (6 sèries):')
display(sub_view[['RMSE','MAE','MAPE','MASE']].round(2))

tot_view = df_metrics_ens[df_metrics_ens['Tecnologia'] == 'Total'].set_index('Tecnologia')
print('\nMètriques Ensemble al Total:')
display(tot_view[['RMSE','MAE','MAPE','MASE']].round(2))

### 1.3 Visualització — backtesting per tecnologia

In [ ]:
SUB_TECHS = [t for t in TECNOLOGIES if t != 'Total']
plot_backtest_per_tech(
    pivot_train, pivot_test, df_forecasts_ens, df_metrics_ens,
    tecnologies=SUB_TECHS, model_name='Ensemble',
    save_path=FIGURES_DIR / 'fig_ensemble_backtest_per_tech.png',
    color_pred='#d73027',
)
plt.show()

In [ ]:
tech = 'Total'
sub  = df_forecasts_ens[df_forecasts_ens['Tecnologia'] == tech].sort_values('date')

fig, ax = plt.subplots(figsize=(13, 4))
y_train_plot   = pivot_train[tech].iloc[-36:]
y_test_plot    = pivot_test[tech]
observed_dates = list(y_train_plot.index) + list(y_test_plot.index)
observed_vals  = list(y_train_plot.values) + list(y_test_plot.values)
ax.plot(observed_dates, observed_vals, color='#2c7bb6', lw=2, label='Observat (historial)')
ax.axvspan(pivot_test.index[0], pivot_test.index[-1], alpha=0.10, color='gray', label='Backtesting (2025)')
ax.axvline(pivot_test.index[0], color='gray', ls=':', lw=1.2)

# Components individuals (traç fi) + Ensemble (traç gruixut)
for fc_df, model_name, color, lw, alpha in [
    (fc_ets, 'ETS Additiu',  '#2166ac', 1.0, 0.6),
    (fc_sar, 'AutoSARIMA',   '#f46d43', 1.0, 0.6),
    (fc_tbt, 'TBATS',        '#7b2d8b', 1.0, 0.6),
]:
    s = fc_df[fc_df['Tecnologia'] == tech].sort_values('date')
    ax.plot(s['date'], s['predicted'], color=color, lw=lw, ls='--', alpha=alpha, label=model_name)

m_total = df_metrics_ens[df_metrics_ens['Tecnologia'] == tech].iloc[0]
label = (f'Ensemble  RMSE={m_total["RMSE"]:.0f}  MAE={m_total["MAE"]:.0f}  '
         f'MAPE={m_total["MAPE"]:.1f}%  MASE={m_total["MASE"]:.2f}')
ax.plot(sub['date'], sub['predicted'], color='#d73027', lw=2.2, ls='--', marker='o', ms=5, label=label)

ax.set_title(f'Ensemble — {tech}: predicció vs. observat (backtesting 2025)', fontsize=13)
ax.set_ylabel('Matriculacions mensuals')
ax.legend(fontsize=8, framealpha=0.9, ncol=2)
format_ts_ax(ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_22_ensemble_total.png', bbox_inches='tight')
plt.show()

### 1.4 Walk-forward validation (rolling-origin)

Per a l'Ensemble, el WFV no requereix cap nou entrenament: simplement combinem les prediccions WFV dels tres components (ja calculades i guardades) amb la mateixa lògica de mitjana aritmètica. Per a cada combinació (Tecnologia, fold, horitzó h), la predicció de l'Ensemble és la mitjana de les prediccions ETS Additiu, AutoSARIMA i TBATS al mateix fold i horitzó. Les mètriques per fold es computen sobre les 12 prediccions del horitzó complet.

In [ ]:
# Carregar WFV forecasts dels 3 components
wfv_baseline = pd.read_csv(RESULTS_DIR / 'baseline_wfv_forecasts.csv', parse_dates=['fold_start', 'date'])
wfv_sarima   = pd.read_csv(RESULTS_DIR / 'sarima_wfv_forecasts.csv',   parse_dates=['fold_start', 'date'])
wfv_tbats_f  = pd.read_csv(RESULTS_DIR / 'tbats_wfv_forecasts.csv',    parse_dates=['fold_start', 'date'])

wfv_ets = wfv_baseline[wfv_baseline['Model'] == 'ETS_Additiu'].copy()
wfv_sar = wfv_sarima[wfv_sarima['Model'] == 'AutoSARIMA'].copy()
wfv_tbt = wfv_tbats_f[wfv_tbats_f['Model'] == 'TBATS'].copy()

key_wfv = ['Tecnologia', 'fold_start', 'h', 'date']
ens_wfv_fc = (wfv_ets[key_wfv + ['predicted', 'actual']]
              .merge(wfv_sar[key_wfv + ['predicted']].rename(columns={'predicted': 'pred_sar'}), on=key_wfv)
              .merge(wfv_tbt[key_wfv + ['predicted']].rename(columns={'predicted': 'pred_tbt'}), on=key_wfv))
ens_wfv_fc['predicted'] = (ens_wfv_fc['predicted'] + ens_wfv_fc['pred_sar'] + ens_wfv_fc['pred_tbt']) / 3
ens_wfv_fc = ens_wfv_fc[key_wfv + ['predicted', 'actual']].copy()
ens_wfv_fc['Model'] = 'Ensemble'

print(f'WFV Ensemble: {len(ens_wfv_fc)} registres '
      f'({ens_wfv_fc["fold_start"].nunique()} folds × {H} horitzons × {len(TECNOLOGIES)} tecns)')

In [ ]:
# Mètriques per fold (necessitem y_train de cada fold per calcular MASE)
wfv_metric_rows = []
for (tech, fold_start), group in ens_wfv_fc.groupby(['Tecnologia', 'fold_start']):
    g = group.sort_values('h')
    y_true = g['actual'].values
    y_pred = g['predicted'].values
    fold_pos = pivot_full.index.get_loc(fold_start)
    y_train  = pivot_full[tech].iloc[:fold_pos].dropna().values
    m = compute_metrics(y_true, y_pred, y_train)
    m.update({'fold_start': fold_start, 'n_train': fold_pos, 'Tecnologia': tech, 'Model': 'Ensemble'})
    wfv_metric_rows.append(m)

df_wfv_metrics_ens  = pd.DataFrame(wfv_metric_rows)
df_wfv_forecasts_ens = ens_wfv_fc.copy()

wfv_agg_ens = (df_wfv_metrics_ens.dropna(subset=['MASE'])
               .groupby('Tecnologia')[['RMSE','MAE','MAPE','MASE']]
               .mean().round(2).reset_index())

print('Mitjana WFV Ensemble per tecnologia:')
display(wfv_agg_ens.set_index('Tecnologia'))
global_mase = df_wfv_metrics_ens.dropna(subset=['MASE'])['MASE'].mean()
print(f'\nMASE global WFV (91 folds): {global_mase:.3f}')

In [ ]:
SUB_TECHS = [t for t in TECNOLOGIES if t != 'Total']
plot_wfv_per_tech(
    pivot_full, df_wfv_forecasts_ens, wfv_agg_ens,
    tecnologies=SUB_TECHS, model_name='Ensemble',
    save_path=FIGURES_DIR / 'fig_23_ensemble_wfv_per_tech.png',
    color_pred='#d73027',
)
plt.show()

In [ ]:
# Guardar resultats Ensemble
df_forecasts_ens.to_csv(RESULTS_DIR / 'ensemble_forecasts.csv', index=False)
df_metrics_ens.to_csv(RESULTS_DIR  / 'ensemble_metrics.csv',   index=False)
df_wfv_metrics_ens.to_csv(RESULTS_DIR  / 'ensemble_wfv_metrics.csv',   index=False)
df_wfv_forecasts_ens.to_csv(RESULTS_DIR / 'ensemble_wfv_forecasts.csv', index=False)
print('Guardats: ensemble_forecasts.csv, ensemble_metrics.csv')
print('          ensemble_wfv_metrics.csv, ensemble_wfv_forecasts.csv')

---
## Secció 2: Global LightGBM (cross-series)

### 2.1 Motivació i disseny

El LightGBM individual del benchmark obtenia un WFV MASE=1.47, clarament per sota del SeasonalNaive (1.30). El motiu és estructural: amb N≈84 mostres per sèrie i estacionalitat anual m=12, el gradient boosting no disposa de prou dades per aprendre l'estructura estacional implícitament. En canvi, els models estadístics capturen l'estacionalitat via paràmetres explícits (coeficients AR, sinus/cosinus de Fourier, suavitzat exponencial), que s'estimen amb molta eficiència amb N petit.

La solució és **compartir la força estadística entre sèries** (*cross-learning* o *global model*). En lloc d'entrenar 7 models separats (un per tecnologia), entrenem un sol model LightGBM sobre la matriu de features concatenada de les 7 sèries: 7 × 84 ≈ 588 observacions. Mantenim l'estratègia *direct multi-step*: un model per a cada horitzó h=1..12.

Per diferenciar les tecnologies, afegim una variable `tech_id` (enter 0–6) marcada com a **categòrica** a LightGBM. Això permet als arbres de decisió partir explícitament per tecnologia quan és útil (per exemple, el pic de Gasolina/Elèctric és diferent del d'Elèctric pur) i compartir representació quan l'estacionalitat és similar.

| Paràmetre | Valor | Raó |
|---|---|---|
| Arquitectura | 1 model per h (direct multi-step) | Evita propagació d'errors recursiva |
| Dades d'entrenament | 7 sèries concatenades | 7× més dades per aprendre estacionalitat |
| Diferenciació de sèrie | `tech_id` categòric (0–6) | LightGBM parteix per tech quan millora |
| Hiperparàmetres | Idèntics al LGBM individual | Comparació justa |

### 2.2 Entrenament — backtest 2025

In [ ]:
# Construir la matriu global de features i els targets per horitzó
all_X_h = {h: [] for h in range(1, H + 1)}
all_y_h = {h: [] for h in range(1, H + 1)}
X_last_per_tech = {}

for i_tech, tech in enumerate(TECNOLOGIES):
    df_tech = train_set[train_set['Tecnologia'] == tech][['date', 'Matriculacions']].copy()
    df_feat = build_ml_features(df_tech)
    df_clean = df_feat.dropna(subset=FEATURE_COLS).copy()
    df_clean['tech_id'] = i_tech

    X = df_clean[FEATURE_COLS_G].values
    y = df_clean['Matriculacions'].values

    for h in range(1, H + 1):
        n_valid = len(y) - h
        if n_valid <= 0:
            continue
        all_X_h[h].append(pd.DataFrame(X[:n_valid], columns=FEATURE_COLS_G))
        all_y_h[h].append(y[h:])

    # Fila de predicció: última fila del train complet (amb tots els lags disponibles)
    X_last = df_feat[FEATURE_COLS].iloc[[-1]].copy()
    X_last['tech_id'] = i_tech
    X_last_per_tech[tech] = X_last[FEATURE_COLS_G].copy()

# Entrenar un model per horitzó
models_global = {}
for h in range(1, H + 1):
    X_h_df = pd.concat(all_X_h[h], ignore_index=True)
    y_h    = np.concatenate(all_y_h[h])
    model  = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(X_h_df, y_h, categorical_feature=['tech_id'])
    models_global[h] = model

n_train_global = len(pd.concat(all_X_h[1], ignore_index=True))
print(f'Matriu global h=1: {n_train_global} observacions × {len(FEATURE_COLS_G)} features')
print(f'Entrenats {len(models_global)} models globals (h=1..{H})')

In [ ]:
# Predicció i mètriques
forecast_records_g = []
metrics_records_g  = []

for tech in TECNOLOGIES:
    y_pred  = np.array([max(models_global[h].predict(X_last_per_tech[tech])[0], 0)
                        for h in range(1, H + 1)])
    y_test  = pivot_test[tech].values
    y_train = pivot_train[tech].dropna().values

    m = compute_metrics(y_test, y_pred, y_train)
    m.update({'Tecnologia': tech, 'Model': 'GlobalLGBM'})
    metrics_records_g.append(m)
    print(f'  {tech:35s}: RMSE={m["RMSE"]:.0f}, MAPE={m["MAPE"]:.1f}%, MASE={m["MASE"]:.2f}')

    for i, date in enumerate(pivot_test.index):
        forecast_records_g.append({'date': date, 'Tecnologia': tech, 'Model': 'GlobalLGBM',
                                   'predicted': y_pred[i], 'actual': y_test[i]})

df_metrics_g   = pd.DataFrame(metrics_records_g)
df_forecasts_g = pd.DataFrame(forecast_records_g)

sub_view_g = df_metrics_g[df_metrics_g['Tecnologia'] != 'Total'].set_index('Tecnologia')
print('\nMètriques Global LightGBM per sub-tecnologia (6 sèries):')
display(sub_view_g[['RMSE','MAE','MAPE','MASE']].round(2))

tot_view_g = df_metrics_g[df_metrics_g['Tecnologia'] == 'Total'].set_index('Tecnologia')
print('\nMètriques Global LightGBM al Total:')
display(tot_view_g[['RMSE','MAE','MAPE','MASE']].round(2))

### 2.3 Feature Importance — h=1 (Global)

In [ ]:
model_h1 = models_global[1]
importance = pd.Series(model_h1.feature_importances_,
                       index=FEATURE_COLS_G).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors_imp = ['#d73027' if f == 'tech_id' else 'steelblue' for f in importance.tail(15).index]
importance.tail(15).plot.barh(ax=ax, color=colors_imp)
ax.set_title('Global LightGBM — Feature Importance (h=1, totes les tecns)', fontsize=12)
ax.set_xlabel('Importance (gain)')
ax.axvline(0, color='black', lw=0.8)
# Anotació tech_id
if 'tech_id' in importance.tail(15).index:
    rank = list(importance.tail(15).index).index('tech_id')
    ax.text(importance['tech_id'] * 1.02, rank, 'tech_id (nou)', va='center', fontsize=8, color='#d73027')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_global_lgbm_importance.png', bbox_inches='tight')
plt.show()

print(f"\nImportància de tech_id: {importance['tech_id']:.0f} "
      f"(posició {len(importance) - importance.rank().astype(int)['tech_id'] + 1} de {len(importance)}")

### 2.4 Visualització — backtesting per tecnologia

In [ ]:
SUB_TECHS = [t for t in TECNOLOGIES if t != 'Total']
plot_backtest_per_tech(
    pivot_train, pivot_test, df_forecasts_g, df_metrics_g,
    tecnologies=SUB_TECHS, model_name='Global LightGBM',
    save_path=FIGURES_DIR / 'fig_global_lgbm_backtest_per_tech.png',
    color_pred='#4dac26',
)
plt.show()

In [ ]:
tech = 'Total'
# Comparació directa: LGBM individual vs Global LGBM
lgbm_ind = pd.read_csv(RESULTS_DIR / 'lgbm_forecasts.csv', parse_dates=['date'])
lgbm_ind = lgbm_ind[lgbm_ind['Tecnologia'] == tech].sort_values('date')

fig, ax = plt.subplots(figsize=(13, 4))
y_train_plot   = pivot_train[tech].iloc[-36:]
y_test_plot    = pivot_test[tech]
observed_dates = list(y_train_plot.index) + list(y_test_plot.index)
observed_vals  = list(y_train_plot.values) + list(y_test_plot.values)
ax.plot(observed_dates, observed_vals, color='#2c7bb6', lw=2, label='Observat (historial)')
ax.axvspan(pivot_test.index[0], pivot_test.index[-1], alpha=0.10, color='gray', label='Backtesting (2025)')
ax.axvline(pivot_test.index[0], color='gray', ls=':', lw=1.2)

lgbm_m = pd.read_csv(RESULTS_DIR / 'lgbm_metrics.csv')
lgbm_m_tot = lgbm_m[lgbm_m['Tecnologia'] == tech].iloc[0]
ax.plot(lgbm_ind['date'], lgbm_ind['predicted'], color='#1a9641', lw=1.3, ls='--',
        label=f'LGBM individual  MASE={lgbm_m_tot["MASE"]:.2f}', alpha=0.7)

sub_g = df_forecasts_g[df_forecasts_g['Tecnologia'] == tech].sort_values('date')
m_g   = df_metrics_g[df_metrics_g['Tecnologia'] == tech].iloc[0]
label = (f'Global LGBM  RMSE={m_g["RMSE"]:.0f}  MAE={m_g["MAE"]:.0f}  '
         f'MAPE={m_g["MAPE"]:.1f}%  MASE={m_g["MASE"]:.2f}')
ax.plot(sub_g['date'], sub_g['predicted'], color='#4dac26', lw=2, ls='--',
        marker='o', ms=5, label=label)

ax.set_title(f'Global LightGBM — {tech}: predicció vs. observat (backtesting 2025)', fontsize=13)
ax.set_ylabel('Matriculacions mensuals')
ax.legend(fontsize=9, framealpha=0.9)
format_ts_ax(ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_24_global_lgbm_total.png', bbox_inches='tight')
plt.show()

### 2.5 Walk-forward validation (rolling-origin)

El WFV del Global LightGBM requereix un bucle personalitzat: a cada fold s'ha de reconstruir la matriu global de features (les 7 sèries concatenades fins al punt d'origin) i reentrenar els 12 models. El cost aproximat és proporcional al del LGBM individual però amb 7× més dades per model: **~15-25 minuts en total** (13 folds × 12 models).

Per cada fold amb origin a `start`:
1. Per a cada tecnologia, construïm features sobre `pivot_full[tech].iloc[:start]`.
2. Concatenem les 7 matrius (amb `tech_id`) per formar la matriu global.
3. Entrenem 12 models (`h=1..12`) sobre la matriu global.
4. Predim els 12 mesos de test (`pivot_full[tech].iloc[start:start+12]`) per a cada tecnologia.

In [ ]:
INITIAL, STEP = 60, 3
n_full = len(pivot_full)
fold_starts = list(range(INITIAL, n_full - H + 1, STEP))
print(f'Folds: {len(fold_starts)} (origins: {pivot_full.index[INITIAL].date()} → {pivot_full.index[fold_starts[-1]].date()})')

wfv_metric_rows_g   = []
wfv_forecast_rows_g = []

t0 = time.time()
for fi, start in enumerate(fold_starts):
    # --- Construir matriu global per a aquest fold ---
    all_X_h_fold = {h: [] for h in range(1, H + 1)}
    all_y_h_fold = {h: [] for h in range(1, H + 1)}
    X_last_fold  = {}

    for i_tech, tech in enumerate(TECNOLOGIES):
        train_s = pivot_full[tech].iloc[:start].dropna()
        df_tech = pd.DataFrame({'date': train_s.index, 'Matriculacions': train_s.values})
        df_feat = build_ml_features(df_tech)
        df_clean = df_feat.dropna(subset=FEATURE_COLS).copy()
        df_clean['tech_id'] = i_tech

        if len(df_clean) < H:
            continue

        X = df_clean[FEATURE_COLS_G].values
        y = df_clean['Matriculacions'].values

        for h in range(1, H + 1):
            n_valid = len(y) - h
            if n_valid <= 0:
                continue
            all_X_h_fold[h].append(pd.DataFrame(X[:n_valid], columns=FEATURE_COLS_G))
            all_y_h_fold[h].append(y[h:])

        # X_last per a la predicció
        X_last_row = df_feat[FEATURE_COLS].iloc[[-1]].copy()
        X_last_row['tech_id'] = i_tech
        X_last_fold[tech] = X_last_row[FEATURE_COLS_G].copy()

    # --- Entrenar models globals per a aquest fold ---
    models_fold = {}
    for h in range(1, H + 1):
        if not all_X_h_fold[h]:
            continue
        X_h_df = pd.concat(all_X_h_fold[h], ignore_index=True)
        y_h    = np.concatenate(all_y_h_fold[h])
        model  = lgb.LGBMRegressor(**LGB_PARAMS)
        model.fit(X_h_df, y_h, categorical_feature=['tech_id'])
        models_fold[h] = model

    # --- Predicció per a cada tecnologia ---
    for tech in TECNOLOGIES:
        if tech not in X_last_fold:
            continue
        test_window = pivot_full[tech].iloc[start:start + H]
        if len(test_window) < H:
            continue

        y_pred = np.array([max(models_fold[h].predict(X_last_fold[tech])[0], 0)
                           for h in range(1, H + 1) if h in models_fold])
        y_true = test_window.values
        y_tr   = pivot_full[tech].iloc[:start].dropna().values

        m = compute_metrics(y_true, y_pred, y_tr)
        m.update({'fold_start': pivot_full.index[start], 'n_train': start,
                  'Tecnologia': tech, 'Model': 'GlobalLGBM'})
        wfv_metric_rows_g.append(m)

        for h_idx in range(H):
            wfv_forecast_rows_g.append({
                'fold_start': pivot_full.index[start],
                'h':          h_idx + 1,
                'date':       test_window.index[h_idx],
                'predicted':  float(y_pred[h_idx]),
                'actual':     float(y_true[h_idx]),
                'Tecnologia': tech,
                'Model':      'GlobalLGBM',
            })

    if fi % 4 == 0 or fi == len(fold_starts) - 1:
        print(f'  Fold {fi+1:2d}/{len(fold_starts)}: origin {pivot_full.index[start].strftime("%Y-%m")} '
              f'— {(time.time()-t0)/60:.1f} min')

df_wfv_metrics_g   = pd.DataFrame(wfv_metric_rows_g)
df_wfv_forecasts_g = pd.DataFrame(wfv_forecast_rows_g)
print(f'\nTotal WFV: {(time.time()-t0)/60:.1f} min')

In [ ]:
wfv_agg_g = (df_wfv_metrics_g.dropna(subset=['MASE'])
             .groupby('Tecnologia')[['RMSE','MAE','MAPE','MASE']]
             .mean().round(2).reset_index())

print('Mitjana WFV Global LightGBM per tecnologia:')
display(wfv_agg_g.set_index('Tecnologia'))

global_mase_g = df_wfv_metrics_g.dropna(subset=['MASE'])['MASE'].mean()
print(f'\nMASE global WFV (91 folds): {global_mase_g:.3f}')

# Comparació ràpida amb LGBM individual
lgbm_wfv_m = pd.read_csv(RESULTS_DIR / 'lgbm_wfv_metrics.csv', parse_dates=['fold_start'])
lgbm_global_mase = lgbm_wfv_m.dropna(subset=['MASE'])['MASE'].mean()
print(f'MASE global WFV LGBM individual (referència): {lgbm_global_mase:.3f}')
print(f'Millora Global vs Individual: {(lgbm_global_mase - global_mase_g)/lgbm_global_mase*100:.1f}%')

In [ ]:
SUB_TECHS = [t for t in TECNOLOGIES if t != 'Total']
plot_wfv_per_tech(
    pivot_full, df_wfv_forecasts_g, wfv_agg_g,
    tecnologies=SUB_TECHS, model_name='Global LightGBM',
    save_path=FIGURES_DIR / 'fig_25_global_lgbm_wfv_per_tech.png',
    color_pred='#4dac26',
)
plt.show()

In [ ]:
# Guardar resultats Global LightGBM
df_forecasts_g.to_csv(RESULTS_DIR  / 'global_lgbm_forecasts.csv',     index=False)
df_metrics_g.to_csv(RESULTS_DIR    / 'global_lgbm_metrics.csv',       index=False)
df_wfv_metrics_g.to_csv(RESULTS_DIR   / 'global_lgbm_wfv_metrics.csv',   index=False)
df_wfv_forecasts_g.to_csv(RESULTS_DIR / 'global_lgbm_wfv_forecasts.csv', index=False)
print('Guardats: global_lgbm_forecasts.csv, global_lgbm_metrics.csv')
print('          global_lgbm_wfv_metrics.csv, global_lgbm_wfv_forecasts.csv')

---
## Conclusions

Aquest notebook genera dos models addicionals — **Ensemble** (mitjana ETS Additiu + AutoSARIMA + TBATS) i **Global LightGBM** (un sol model amb totes les sèries i `tech_id` com a feature) — i guarda les seves prediccions a CSV. La comparació global amb la resta de models del benchmark es fa a `evaluation.ipynb`, que autodescobreix els `*_metrics.csv` i els integra al rànquing global del Total i a l'anàlisi per sub-tecnologia.

### Lectura específica dels dos models nous

**Ensemble (ETS Additiu + AutoSARIMA + TBATS):** la combinació simple per mitjana aritmètica suavitza els errors individuals però només n'aprofita el benefici si els components fan errors no correlacionats. En aquest cas els tres comparteixen estructura paramètrica sobre la mateixa sèrie mensual, així que tendeixen a subestimar alhora els pics de 2025: la combinació redueix variància però no biaix sistemàtic. L'avantatge pràctic és modest respecte el millor component individual.

**Global LightGBM (cross-series):** l'enfocament global té sentit teòric (7× més dades d'entrenament) però xoca amb l'**heterogeneïtat d'escala** entre les sèries — del ~20 unitats/mes del Diesel/Elèctric Endollable als ~6.000 del Total. Sense normalització per escala abans de concatenar, els arbres de decisió s'orienten cap als residus de les sèries grans i ignoren la dinàmica de les petites. La millora natural per reactivar el potencial del cross-learning seria normalitzar cada sèrie (z-score o ratio respecte la mitjana) abans d'entrenar, i avaluar amb MAPE en lloc de MSE.

Les conclusions sobre **quin model triar per a producció** i la comparació detallada amb la resta del benchmark són a `evaluation.ipynb`.